In [76]:
%pip install -q tokenizers huggingface_hub datasets deepspeed

Note: you may need to restart the kernel to use updated packages.


In [77]:
import os
os.environ['HF_HOME'] = '/kaggle/working/hf_hub'

import gc
import json
import math
import random
import time
import uuid
from contextlib import nullcontext
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import DataLoader
from tokenizers import Tokenizer
from huggingface_hub import HfApi, hf_hub_download
from kaggle_secrets import UserSecretsClient
from datasets import load_dataset
from tqdm.auto import tqdm
import deepspeed
from deepspeed import comm as ds_dist
from deepspeed.moe.layer import MoE

In [78]:
repo_id = 'MrZVIL/hw2-cg-prenorm_rmsnorm_swiglu_relative_moe_encdec'
experiment_name = 'prenorm_rmsnorm_swiglu_relative_moe_encdec'
checkpoint_prefix = 'training/moe_40m_stage2'
checkpoint_filename = 'last.pt'
inference_batch_size = 16
num_beams = 30
max_new_tokens = 64
length_penalty = 0.6
use_kv_cache = True
use_amp = True

assert torch.cuda.is_available(), 'This notebook requires a CUDA GPU'
torch.cuda.set_device(0)
device = torch.device('cuda:0')
os.environ.setdefault('LOCAL_RANK', '0')

if not ds_dist.is_initialized():
    deepspeed.init_distributed(
        dist_backend='nccl',
        auto_mpi_discovery=False,
        init_method=f'file:///tmp/moe_inference_{uuid.uuid4().hex}',
        rank=0,
        world_size=1,
    )
assert ds_dist.get_world_size() == 1

hf_token = UserSecretsClient().get_secret('huggingface-cli')
api = HfApi(token=hf_token)


def amp_context():
    return (
        torch.autocast(device_type='cuda', dtype=torch.float16)
        if use_amp else nullcontext()
    )

In [79]:
revision = api.model_info(repo_id).sha

checkpoint_path = hf_hub_download(
    repo_id=repo_id,
    filename=f'{checkpoint_prefix}/{checkpoint_filename}',
    revision=revision,
    token=hf_token,
)
tokenizer_path = hf_hub_download(
    repo_id=repo_id,
    filename=f'{checkpoint_prefix}/tokenizer.json',
    revision=revision,
    token=hf_token,
)

checkpoint = torch.load(checkpoint_path, map_location='cpu', weights_only=True)
config = dict(checkpoint['config'])
checkpoint_step = checkpoint.get('global_step', 0)
checkpoint_epoch = checkpoint.get('epoch')
checkpoint_batch = checkpoint.get('batch_idx')
checkpoint_status = checkpoint.get('status', 'unknown')
tokenizer = Tokenizer.from_file(tokenizer_path)
PAD_ID = tokenizer.token_to_id('[PAD]')
BOS_ID = tokenizer.token_to_id('[BOS]')
EOS_ID = tokenizer.token_to_id('[EOS]')
assert all(x is not None for x in (PAD_ID, BOS_ID, EOS_ID))
assert PAD_ID == checkpoint['pad_id']
assert tokenizer.get_vocab_size() == config['vocab_size']

random.seed(config['seed'])
np.random.seed(config['seed'])
torch.manual_seed(config['seed'])
torch.cuda.manual_seed_all(config['seed'])
print('Checkpoint:', f'{repo_id}/{checkpoint_prefix}/{checkpoint_filename}')
print('Revision:', revision)
print('Epoch:', checkpoint_epoch, 'Batch:', checkpoint_batch, 'Step:', checkpoint_step)
print('Training status at save:', checkpoint_status)
print('Training world size:', checkpoint.get('world_size', config.get('world_size', 1)))
print('Best validation loss:', checkpoint.get('best_val_loss', checkpoint.get('val_loss')))

Checkpoint: MrZVIL/hw2-cg-prenorm_rmsnorm_swiglu_relative_moe_encdec/training/moe_40m_stage2/last.pt
Revision: 7726fb8ceedc274550acaf79f6d0881ef1f79d35
Epoch: 2 Batch: 0 Step: 468549
Training status at save: unknown
Training world size: 1
Best validation loss: 0.2670641683888042


In [80]:
class BasicFeedForward(nn.Module):
    def __init__(self, config):
        super().__init__()

        self.linear1 = nn.Linear(
            config['d_model'],
            config['dim_feedforward'],
        )
        self.linear2 = nn.Linear(
            config['dim_feedforward'],
            config['d_model'],
        )
        self.dropout = nn.Dropout(config['dropout'])
    def forward(self, x):
        return self.linear2(self.dropout(F.relu(self.linear1(x))))

class SwiGLUFeedForward(nn.Module):
    def __init__(self, config):
        super().__init__()

        d_model = config["d_model"]
        hidden_dim = config["dim_feedforward"]

        self.gate_proj = nn.Linear(d_model, hidden_dim)
        self.up_proj = nn.Linear(d_model, hidden_dim)
        self.down_proj = nn.Linear(hidden_dim, d_model)
        self.dropout = nn.Dropout(config["dropout"])

    def forward(self, x):
        hidden = F.silu(self.gate_proj(x)) * self.up_proj(x)
        return self.down_proj(self.dropout(hidden))

class DenseFeedForward(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.ffn = SwiGLUFeedForward(config)

    def forward(self, x, padding_mask=None):
        output = self.ffn(x)
        aux_loss = output.new_zeros((), dtype=torch.float32)
        return output, aux_loss, None

In [81]:
class MoEFeedForward(nn.Module):
    def __init__(self, config):
        super().__init__()

        self.num_experts = config["num_experts"]

        self.moe = MoE(
            hidden_size=config["d_model"],
            expert=SwiGLUFeedForward(config),
            num_experts=self.num_experts,
            ep_size=1,
            k=config["moe_top_k"],
            drop_tokens=False,
            use_residual=False,
            noisy_gate_policy=None,
            top2_2nd_expert_sampling=False,
            use_tutel=False,
        )

        self.moe.set_deepspeed_parallelism()

    def forward(self, x, padding_mask=None):
        flat = x.reshape(-1, x.size(-1))

        valid = (
            torch.ones(
                flat.size(0),
                dtype=torch.bool,
                device=x.device,
            )
            if padding_mask is None
            else ~padding_mask.reshape(-1)
        )

        indices = valid.nonzero(as_tuple=True)[0]
        tokens = flat[indices]

        if tokens.size(0) == 0:
            return (
                torch.zeros_like(x),
                x.new_zeros((), dtype=torch.float32),
                x.new_zeros(self.num_experts, dtype=torch.float32),
            )

        output, aux_loss, counts = self.moe(tokens)

        result = torch.zeros_like(flat).index_copy(
            0,
            indices,
            output.to(flat.dtype),
        )

        counts = counts.detach().to(
            device=x.device,
            dtype=torch.float32,
        )
        load = counts / counts.sum().clamp_min(1)

        return result.view_as(x), aux_loss, load

In [82]:
class RelativePositionBias(nn.Module):
    def __init__(self, config):
        super().__init__()

        self.max_distance = config["max_relative_distance"]
        self.num_heads = config["nhead"]

        self.embedding = nn.Embedding(
            2 * self.max_distance + 1,
            self.num_heads,
        )

    def forward(self, x, padding_mask=None, causal_mask=None):
        batch_size, length, _ = x.shape
        positions = torch.arange(length, device=x.device)

        distances = positions[None, :] - positions[:, None]
        indices = (
            distances.clamp(-self.max_distance, self.max_distance)
            + self.max_distance
        )

        bias = self.embedding(indices).permute(2, 0, 1)
        bias = bias.to(dtype=x.dtype)
        bias = bias.unsqueeze(0).expand(
            batch_size, -1, -1, -1
        )

        if padding_mask is not None:
            bias = bias.masked_fill(
                padding_mask[:, None, None, :],
                float("-inf"),
            )

        if causal_mask is not None:
            bias = bias.masked_fill(
                causal_mask[None, None, :, :],
                float("-inf"),
            )

        return bias.reshape(
            batch_size * self.num_heads,
            length,
            length,
        )

In [83]:
def cached_attention(attention, query, key_value, past_kv=None,
                     static_kv=False, attention_bias=None):
    batch_size, query_length, width = query.shape
    heads = attention.num_heads
    head_dim = width // heads
    wq, wk, wv = attention.in_proj_weight.chunk(3, dim=0)
    if attention.in_proj_bias is None:
        bq = bk = bv = None
    else:
        bq, bk, bv = attention.in_proj_bias.chunk(3, dim=0)

    def split_heads(x):
        return x.reshape(batch_size, -1, heads, head_dim).transpose(1, 2)

    q = split_heads(F.linear(query, wq, bq))
    if static_kv and past_kv is not None:
        k, v = past_kv
    else:
        k = split_heads(F.linear(key_value, wk, bk))
        v = split_heads(F.linear(key_value, wv, bv))
        if past_kv is not None:
            k = torch.cat([past_kv[0], k], dim=2)
            v = torch.cat([past_kv[1], v], dim=2)

    if attention_bias is not None:
        attention_bias = attention_bias.to(dtype=q.dtype)

    output = F.scaled_dot_product_attention(
        q, k, v,
        attn_mask=attention_bias,
        dropout_p=0.0,
        is_causal=False,
    )
    output = output.transpose(1, 2).reshape(batch_size, query_length, width)
    return attention.out_proj(output), (k, v)


def reorder_cache(cache, indices):
    return [
        {
            name: tuple(value.index_select(0, indices) for value in pair)
            for name, pair in layer_cache.items()
        }
        for layer_cache in cache
    ]

In [84]:
class EncoderLayer(nn.Module):
    def __init__(self, config, use_moe=False):
        super().__init__()

        self.position_bias = RelativePositionBias(config)
        self.self_attn = nn.MultiheadAttention(
            embed_dim=config['d_model'],
            num_heads=config['nhead'],
            dropout=config['dropout'],
            batch_first=True,
        )

        self.use_moe = use_moe
        self.ffn = (
            MoEFeedForward(config)
            if use_moe
            else DenseFeedForward(config)
        )
        self.norm1 = nn.RMSNorm(
            config['d_model'],
            eps=config['norm_eps']
        )
        self.norm2 = nn.RMSNorm(
            config['d_model'],
            eps=config['norm_eps']
        )
        self.dropout1 = nn.Dropout(config['dropout'])
        self.dropout2 = nn.Dropout(config['dropout'])

    def attention(self, x, padding_mask):
        attention_bias = self.position_bias(
            x,
            padding_mask=padding_mask,
        )

        output, _ = self.self_attn(
            query=x,
            key=x,
            value=x,
            attn_mask=attention_bias,
            need_weights=False,
        )
        return output

    def forward(self, x, padding_mask=None):
        x = x + self.dropout1(
            self.attention(self.norm1(x), padding_mask)
        )

        ffn_output, aux_loss, expert_load = self.ffn(
            self.norm2(x),
            padding_mask=padding_mask,
        )

        x = x + self.dropout2(ffn_output)

        return x, aux_loss, expert_load

class DecoderLayer(nn.Module):
    def __init__(self, config, use_moe=False):
        super().__init__()

        self.position_bias = RelativePositionBias(config)
        self.cross_attn = nn.MultiheadAttention(
            embed_dim=config['d_model'],
            num_heads=config['nhead'],
            dropout=config['dropout'],
            batch_first=True,
        )

        self.self_attn = nn.MultiheadAttention(
            embed_dim=config['d_model'],
            num_heads=config['nhead'],
            dropout=config['dropout'],
            batch_first=True,
        )

        self.use_moe = use_moe
        self.ffn = (
            MoEFeedForward(config)
            if use_moe
            else DenseFeedForward(config)
        )

        self.norm1 = nn.RMSNorm(
            config['d_model'],
            eps=config['norm_eps']
        )
        self.norm2 = nn.RMSNorm(
            config['d_model'],
            eps=config['norm_eps']
        )
        self.norm3 = nn.RMSNorm(
            config['d_model'],
            eps=config['norm_eps']
        )

        self.dropout1 = nn.Dropout(config["dropout"])
        self.dropout2 = nn.Dropout(config["dropout"])
        self.dropout3 = nn.Dropout(config["dropout"])

    def self_attention(self, x, causal_mask, padding_mask):
        attention_bias = self.position_bias(
            x,
            padding_mask=padding_mask,
            causal_mask=causal_mask,
        )

        output, _ = self.self_attn(
            query=x,
            key=x,
            value=x,
            attn_mask=attention_bias,
            need_weights=False,
        )
        return output

    def cross_attention(self, x, memory, memory_padding_mask):
        output, _ = self.cross_attn(
            query=x,
            key=memory,
            value=memory,
            key_padding_mask=memory_padding_mask,
            need_weights=False,
        )
        return output

    def forward(
        self,
        x,
        memory,
        causal_mask,
        padding_mask=None,
        memory_padding_mask=None,
    ):
        x = x + self.dropout1(
            self.self_attention(self.norm1(x), causal_mask, padding_mask)
        )
        x = x + self.dropout2(
            self.cross_attention(self.norm2(x), memory, memory_padding_mask)
        )

        ffn_output, aux_loss, expert_load = self.ffn(
            self.norm3(x),
            padding_mask=padding_mask,
        )
        x = x + self.dropout3(ffn_output)

        return x, aux_loss, expert_load


    def step(self, x, memory, memory_padding_mask=None, cache=None):
        if self.training:
            raise RuntimeError('KV-cache requires eval mode')
        if x.size(1) != 1:
            raise ValueError('step expects exactly one new token')

        self_cache = None if cache is None else cache['self']
        cross_cache = None if cache is None else cache['cross']
        past_length = 0 if self_cache is None else self_cache[0].size(2)

        distances = torch.arange(past_length + 1, device=x.device) - past_length
        distance_limit = self.position_bias.max_distance
        indices = distances.clamp(-distance_limit, distance_limit) + distance_limit
        bias = self.position_bias.embedding(indices).transpose(0, 1)[None, :, None, :]

        normalized = self.norm1(x)
        output, self_cache = cached_attention(
            self.self_attn, normalized, normalized,
            past_kv=self_cache, attention_bias=bias,
        )
        x = x + self.dropout1(output)

        cross_bias = None
        if memory_padding_mask is not None:
            cross_bias = x.new_zeros((x.size(0), 1, 1, memory.size(1)))
            cross_bias = cross_bias.masked_fill(
                memory_padding_mask[:, None, None, :], float('-inf')
            )

        output, cross_cache = cached_attention(
            self.cross_attn, self.norm2(x), memory,
            past_kv=cross_cache, static_kv=True, attention_bias=cross_bias,
        )
        x = x + self.dropout2(output)
        ffn_output, _, _ = self.ffn(self.norm3(x))
        x = x + self.dropout3(ffn_output)
        return x, {'self': self_cache, 'cross': cross_cache}

In [85]:
class CodeTransformer(nn.Module):
    def __init__(self, config, pad_id):
        super().__init__()

        self.pad_id = pad_id
        self.d_model = config["d_model"]

        self.token_embedding = nn.Embedding(
            config["vocab_size"],
            self.d_model,
            padding_idx=pad_id,
        )
        self.num_experts = config["num_experts"]
        self.max_src_len = config["max_src_len"]
        self.max_tgt_len = config["max_tgt_len"]

        self.embedding_dropout = nn.Dropout(config["dropout"])

        self.encoder_layers = nn.ModuleList([
            EncoderLayer(
                config,
                use_moe=i in config["encoder_moe_layers"],
            )
            for i in range(config["num_encoder_layers"])
        ])

        self.decoder_layers = nn.ModuleList([
            DecoderLayer(
                config,
                use_moe=i in config["decoder_moe_layers"],
            )
            for i in range(config["num_decoder_layers"])
        ])

        self.encoder_norm = nn.RMSNorm(
            config['d_model'],
            eps=config['norm_eps']
        )
        self.decoder_norm = nn.RMSNorm(
            config['d_model'],
            eps=config['norm_eps']
        )

        self.output = nn.Linear(
            self.d_model,
            config["vocab_size"],
        )

        self.reset_parameters()

    def reset_parameters(self):
        for layers in (self.encoder_layers, self.decoder_layers):
            for parameter in layers.parameters():
                if parameter.dim() > 1:
                    nn.init.xavier_uniform_(parameter)

        nn.init.normal_(self.token_embedding.weight, std=0.02)

        for module in self.modules():
            if isinstance(module, RelativePositionBias):
                nn.init.zeros_(module.embedding.weight)

        with torch.no_grad():
            self.token_embedding.weight[self.pad_id].zero_()

    def collect_aux(self, x, aux_losses, expert_loads):
        if not aux_losses:
            return (
                x.new_zeros((), dtype=torch.float32),
                x.new_zeros((0, self.num_experts), dtype=torch.float32),
            )

        return (
            torch.stack(aux_losses).mean(),
            torch.stack(expert_loads),
        )

    def embed(self, ids):
        x = self.token_embedding(ids) * math.sqrt(self.d_model)
        return self.embedding_dropout(x)

    def encode(self, src, return_aux=False):
        padding_mask = src.eq(self.pad_id)
        memory = self.embed(src)

        aux_losses = []
        expert_loads = []

        for layer in self.encoder_layers:
            memory, aux_loss, expert_load = layer(
                memory,
                padding_mask=padding_mask,
            )

            if return_aux and layer.use_moe:
                aux_losses.append(aux_loss)
                expert_loads.append(expert_load)

        memory = self.encoder_norm(memory)

        if return_aux:
            aux_loss, loads = self.collect_aux(
                memory, aux_losses, expert_loads
            )
            return memory, aux_loss, loads

        return memory


    def decode(
        self,
        tgt_input,
        memory,
        memory_padding_mask=None,
        return_aux=False,
    ):
        length = tgt_input.size(1)

        causal_mask = torch.triu(
            torch.ones(
                length,
                length,
                dtype=torch.bool,
                device=tgt_input.device,
            ),
            diagonal=1,
        )

        padding_mask = tgt_input.eq(self.pad_id)
        x = self.embed(tgt_input)

        aux_losses = []
        expert_loads = []

        for layer in self.decoder_layers:
            x, aux_loss, expert_load = layer(
                x,
                memory,
                causal_mask=causal_mask,
                padding_mask=padding_mask,
                memory_padding_mask=memory_padding_mask,
            )

            if return_aux and layer.use_moe:
                aux_losses.append(aux_loss)
                expert_loads.append(expert_load)

        logits = self.output(self.decoder_norm(x))

        if return_aux:
            aux_loss, loads = self.collect_aux(
                x, aux_losses, expert_loads
            )
            return logits, aux_loss, loads

        return logits


    def forward(self, src, tgt_input, return_aux=False):
        if not return_aux:
            memory = self.encode(src)
            return self.decode(
                tgt_input,
                memory,
                memory_padding_mask=src.eq(self.pad_id),
            )

        memory, encoder_aux, encoder_loads = self.encode(
            src,
            return_aux=True,
        )

        logits, decoder_aux, decoder_loads = self.decode(
            tgt_input,
            memory,
            memory_padding_mask=src.eq(self.pad_id),
            return_aux=True,
        )

        n_encoder = encoder_loads.size(0)
        n_decoder = decoder_loads.size(0)

        aux_loss = (
            encoder_aux * n_encoder + decoder_aux * n_decoder
        ) / max(n_encoder + n_decoder, 1)

        expert_loads = torch.cat(
            [encoder_loads, decoder_loads],
            dim=0,
        )

        return logits, aux_loss, expert_loads

    @torch.inference_mode()
    def decode_step(self, token_ids, memory, memory_padding_mask=None, cache=None):
        if self.training:
            raise RuntimeError('KV-cache requires eval mode')
        if token_ids.ndim != 2 or token_ids.size(1) != 1:
            raise ValueError('decode_step expects [batch, 1] tokens without padding')
        if cache is None:
            cache = [None] * len(self.decoder_layers)
        if len(cache) != len(self.decoder_layers):
            raise ValueError('Invalid number of cache layers')

        x = self.embed(token_ids)
        new_cache = []
        for layer, layer_cache in zip(self.decoder_layers, cache):
            x, layer_cache = layer.step(
                x, memory, memory_padding_mask=memory_padding_mask, cache=layer_cache
            )
            new_cache.append(layer_cache)
        return self.output(self.decoder_norm(x)), new_cache

In [86]:
model = CodeTransformer(config, pad_id=PAD_ID)
model.load_state_dict(checkpoint['model_state_dict'], strict=True)
model = model.to(device).eval()
print('Parameters:', sum(p.numel() for p in model.parameters()))
print('DeepSpeed:', deepspeed.__version__)
print('PyTorch:', torch.__version__)
del checkpoint
gc.collect()

Parameters: 40468348
DeepSpeed: 0.19.8
PyTorch: 2.11.0+cu128


16

In [87]:
@torch.inference_mode()
def generate_batch(
    model,
    tokenizer,
    questions,
    max_src_len,
    max_new_tokens=64,
    num_beams=1,
    length_penalty=0.6,
    use_cache=True,
):
    if not questions:
        return []

    if num_beams < 1:
        raise ValueError("num_beams must be positive")
    if not 1 <= max_src_len <= model.max_src_len:
        raise ValueError("Invalid max_src_len")
    if not 1 <= max_new_tokens <= model.max_tgt_len:
        raise ValueError("Invalid max_new_tokens")
    if length_penalty < 0:
        raise ValueError("length_penalty must be non-negative")

    device = next(model.parameters()).device
    model.eval()

    pad_id = tokenizer.token_to_id("[PAD]")
    bos_id = tokenizer.token_to_id("[BOS]")
    eos_id = tokenizer.token_to_id("[EOS]")

    if any(token_id is None for token_id in (pad_id, bos_id, eos_id)):
        raise ValueError("Missing PAD, BOS or EOS token")

    forbidden_ids = sorted({
        tokenizer.token_to_id(token)
        for token in ["[PAD]", "[BOS]", "[UNK]"]
        if tokenizer.token_to_id(token) is not None
    })

    encoded = tokenizer.encode_batch(questions)
    src = pad_sequence(
        [
            torch.tensor(
                item.ids[:max_src_len - 1] + [eos_id],
                dtype=torch.long,
            )
            for item in encoded
        ],
        batch_first=True,
        padding_value=pad_id,
    ).to(device)

    batch_size = src.size(0)
    src_padding = src.eq(pad_id)
    memory = model.encode(src)

    memory = memory.repeat_interleave(num_beams, dim=0)
    src_padding = src_padding.repeat_interleave(num_beams, dim=0)

    sequences = torch.full(
        (batch_size, num_beams, 1),
        bos_id,
        dtype=torch.long,
        device=device,
    )

    scores = torch.full(
        (batch_size, num_beams),
        float("-inf"),
        device=device,
    )
    scores[:, 0] = 0.0

    lengths = torch.zeros(
        batch_size, num_beams,
        dtype=torch.long,
        device=device,
    )
    finished = torch.zeros(
        batch_size, num_beams,
        dtype=torch.bool,
        device=device,
    )

    cache = None

    for _ in range(max_new_tokens):
        if use_cache:
            decoder_input = sequences[:, :, -1].reshape(batch_size * num_beams, 1)
            step_logits, cache = model.decode_step(
                decoder_input, memory,
                memory_padding_mask=src_padding, cache=cache,
            )
            logits = step_logits[:, -1].float()
        else:
            decoder_input = sequences.reshape(batch_size * num_beams, -1)
            logits = model.decode(
                decoder_input, memory, memory_padding_mask=src_padding,
            )[:, -1].float()

        vocab_size = logits.size(-1)

        if num_beams > vocab_size - len(forbidden_ids):
            raise ValueError("num_beams exceeds allowed vocabulary size")

        logits[:, forbidden_ids] = float("-inf")

        log_probs = logits.log_softmax(dim=-1).reshape(
            batch_size, num_beams, vocab_size
        )

        log_probs = log_probs.masked_fill(
            finished.unsqueeze(-1),
            float("-inf"),
        )
        log_probs[:, :, eos_id] = torch.where(
            finished,
            torch.zeros_like(scores),
            log_probs[:, :, eos_id],
        )

        candidate_scores = scores.unsqueeze(-1) + log_probs
        candidate_lengths = lengths + (~finished).long()

        normalized_scores = candidate_scores / (
            candidate_lengths.clamp_min(1)
            .float()
            .unsqueeze(-1)
            .pow(length_penalty)
        )

        _, best_indices = normalized_scores.reshape(
            batch_size, -1
        ).topk(num_beams, dim=-1)

        parent_indices = best_indices // vocab_size
        next_tokens = best_indices % vocab_size

        if use_cache:
            parent_rows = (
                torch.arange(batch_size, device=device)[:, None] * num_beams
                + parent_indices
            ).reshape(-1)
            cache = reorder_cache(cache, parent_rows)

        scores = candidate_scores.reshape(
            batch_size, -1
        ).gather(1, best_indices)

        lengths = candidate_lengths.gather(1, parent_indices)
        finished = (
            finished.gather(1, parent_indices)
            | next_tokens.eq(eos_id)
        )

        sequences = sequences.gather(
            1,
            parent_indices.unsqueeze(-1).expand(
                -1, -1, sequences.size(-1)
            ),
        )
        sequences = torch.cat(
            [sequences, next_tokens.unsqueeze(-1)],
            dim=-1,
        )

        if finished.all().item():
            break

    final_scores = scores / (
        lengths.clamp_min(1).float().pow(length_penalty)
    )
    best_beams = final_scores.argmax(dim=-1)

    best_sequences = sequences[
        torch.arange(batch_size, device=device),
        best_beams,
    ]

    answers = []
    for ids in best_sequences[:, 1:].cpu().tolist():
        if eos_id in ids:
            ids = ids[:ids.index(eos_id)]

        answers.append(
            tokenizer.decode(ids, skip_special_tokens=True)
        )

    return answers

In [88]:
@torch.inference_mode()
def check_kv_cache(model):
    model.eval()
    normal_ids = [i for i in range(config['vocab_size']) if i not in (PAD_ID, BOS_ID, EOS_ID)]
    a, b, c, d = normal_ids[:4]
    src = torch.tensor([[a, b, EOS_ID, PAD_ID], [a, c, d, EOS_ID]], device=device)
    target = torch.tensor([[BOS_ID, a, b, c], [BOS_ID, d, c, b]], device=device)
    padding = src.eq(PAD_ID)
    with torch.autocast(device_type=device.type, enabled=False):
        memory = model.encode(src)
        expected = model.decode(target, memory, memory_padding_mask=padding)
        cache = None
        outputs = []
        for index in range(target.size(1)):
            logits, cache = model.decode_step(
                target[:, index:index + 1], memory,
                memory_padding_mask=padding, cache=cache,
            )
            outputs.append(logits)
        actual = torch.cat(outputs, dim=1)
        torch.testing.assert_close(actual, expected, rtol=1e-4, atol=1e-4)

        order = torch.tensor([1, 0, 1], device=device)
        cache = reorder_cache(cache, order)
        next_ids = torch.tensor([[a], [b], [c]], device=device)
        actual, _ = model.decode_step(
            next_ids, memory.index_select(0, order),
            memory_padding_mask=padding.index_select(0, order), cache=cache,
        )
        prefix = torch.cat([target.index_select(0, order), next_ids], dim=1)
        expected = model.decode(
            prefix, memory.index_select(0, order),
            memory_padding_mask=padding.index_select(0, order),
        )[:, -1:]
        torch.testing.assert_close(actual, expected, rtol=1e-4, atol=1e-4)
    print('KV-cache matches full-prefix decoding, including beam reordering')

check_kv_cache(model)

KV-cache matches full-prefix decoding, including beam reordering


In [89]:
def test_collate_fn(examples):
    return {'question': [example['question'] for example in examples]}

test_dataset = load_dataset('SlavaYus/tinysms_qa_12m', split='test')
assert len(test_dataset) == 1919

test_loader = DataLoader(
    test_dataset,
    batch_size=inference_batch_size,
    shuffle=False,
    collate_fn=test_collate_fn,
    num_workers=0,
)

Resolving data files:   0%|          | 0/60 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/60 [00:00<?, ?it/s]

In [90]:
output_path = Path('/kaggle/working') / f'submission_{experiment_name}_step{checkpoint_step}.jsonl'
temporary_path = output_path.with_suffix('.jsonl.tmp')
written = 0
torch.cuda.synchronize()
started = time.perf_counter()

with temporary_path.open('w', encoding='utf-8') as f:
    for tasks in tqdm(test_loader, desc='Generating'):
        with amp_context():
            codes = generate_batch(
                model=model,
                tokenizer=tokenizer,
                questions=tasks['question'],
                max_src_len=config['max_src_len'],
                max_new_tokens=max_new_tokens,
                num_beams=num_beams,
                length_penalty=length_penalty,
                use_cache=use_kv_cache,
            )
        assert len(codes) == len(tasks['question'])
        for code in codes:
            f.write(json.dumps({'code': code}, ensure_ascii=False) + '\n')
            written += 1

assert written == len(test_dataset)
os.replace(temporary_path, output_path)
torch.cuda.synchronize()
print('Saved:', output_path)
print('Programs:', written)
print('Generation seconds:', time.perf_counter() - started)

Generating:   0%|          | 0/120 [00:00<?, ?it/s]

Saved: /kaggle/working/submission_prenorm_rmsnorm_swiglu_relative_moe_encdec_step468549.jsonl
Programs: 1919
Generation seconds: 148.01366662099963
